# Notebook 4:
### BI Aggregations, KPI Layer, and Dimensional Outputs

**Purpose**

Builds the Power BI semantic layer, including KPI summaries, date dimension, revenue aggregations, and cross table harmonization required for dashboard reporting.

#### BI outputs
- `date_table.csv`
- `kpi_summary.csv`
- `prescription_volume_by_day.csv`
- `prescription_volume_by_class_day.csv`
- `forecast_demand_by_class_day.csv`
- `revenue_by_insurance_type.csv`
- `inventory_turnover.csv`

#### Reference outputs
- `drug_class_map.csv`
- `ref_drugname_class_map.csv`

**Key responsibilitiy**
- Prepares clean, analysis ready fact tables and KPI summaries for the Power BI reporting layer.

In [4]:
# --- Config (paths) ---
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT  # core synthetic CSVs live here by default
OUT_DIR = PROJECT_ROOT / "bi_outputs"   # Power BI upload folder
OUT_BI = Path("bi_outputs")
OUT_REF = Path("reference_outputs")
FORECAST_DIR = PROJECT_ROOT / "forecast"  # optional intermediate outputs

OUT_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

OUT_BI.mkdir(exist_ok=True)
OUT_REF.mkdir(exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR       =", OUT_DIR)
print("FORECAST_DIR  =", FORECAST_DIR)


PROJECT_ROOT = /Users/selenadavis/PythonProject/Notebooks
OUT_DIR       = /Users/selenadavis/PythonProject/Notebooks/bi_outputs
FORECAST_DIR  = /Users/selenadavis/PythonProject/Notebooks/forecast


In [5]:
# --- Imports, paths, and helpers ---
import math
from pathlib import Path

import numpy as np
import pandas as pd

OUT_DIR = Path("bi_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_DIR = Path("forecast")

def first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def rename_any(df, candidates, target):
    for c in candidates:
        if c in df.columns and target not in df.columns:
            return df.rename(columns={c: target})
    return df

import re
def norm_name(x):
    if pd.isna(x):
        return ""
    s = str(x).lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 ]+", "", s)
    s = re.sub(r"\b\d+(\.\d+)?\b", "", s)
    s = re.sub(r"\b(mg|mcg|g|ml|tab|tabs|cap|caps|unit|units|tablet|tablets|capsule|capsules|solution|inhaler)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Core inputs
P_PATIENTS = first_existing("patients.csv")
P_RX       = first_existing("prescriptions.csv")
P_INV      = first_existing("inventory.csv")
P_REF      = first_existing("ref_drugs.csv")  # optional

# Forecast inputs
P_DAILY_CLASS = first_existing(
    FORECAST_DIR / "predictions_daily_class.csv",
    "predictions_daily_class.csv",
    FORECAST_DIR / "daily_demand_by_class.csv",
    "daily_demand_by_class.csv",
)

P_FCST_6M = first_existing(
    FORECAST_DIR / "forecast_6m_by_class.csv",
    "forecast_6m_by_class.csv",
)

# Staffing inputs
P_STAFF_HOURLY = first_existing(
    FORECAST_DIR / "projected_staffing_by_hour.csv",
    "projected_staffing_by_hour.csv",
)

P_STAFF_COVERAGE = first_existing(
    FORECAST_DIR / "staffing_coverage.csv",
    "staffing_coverage.csv",
)

print("Inputs found:")
print(" - prescriptions:", P_RX)
print(" - patients:", P_PATIENTS)
print(" - inventory:", P_INV)
print(" - ref_drugs:", P_REF)
print(" - daily class forecast:", P_DAILY_CLASS)
print(" - 6m class forecast:", P_FCST_6M)

Inputs found:
 - prescriptions: prescriptions.csv
 - patients: patients.csv
 - inventory: inventory.csv
 - ref_drugs: ref_drugs.csv
 - daily class forecast: forecast/predictions_daily_class.csv
 - 6m class forecast: forecast/forecast_6m_by_class.csv


In [7]:
# --- Build DrugID -> pharm_classes mapping based on forecast distribution (stable) ---
import pandas as pd
import numpy as np
from pathlib import Path

# Output directory
PROJECT_ROOT = Path(".").resolve()

pred = pd.read_csv("predictions_daily_class.csv")
rx = pd.read_csv("prescriptions.csv")
rx["Date"] = pd.to_datetime(rx["FillDate"], errors="coerce").dt.normalize()

daily_actual = (
    rx.groupby("Date", as_index=False)
      .agg(ActualRx=("RxID", "count"))
)

# Standardize forecast schema
pred_out = pred.rename(
    columns={"DrugClass": "pharm_classes", "Demand": "ForecastQty"}
).copy()

# Remove 'Unmapped' from weighting
class_weights = (
    pred_out[pred_out["pharm_classes"] != "Unmapped"]
    .groupby("pharm_classes")["ForecastQty"]
    .sum()
)

classes = class_weights.index.tolist()
probs = (class_weights / class_weights.sum()).to_numpy()

# Clean DrugID
def clean_id(s):
    s = str(s)
    s = s.replace("\u200b", "").replace("\ufeff", "").strip()
    s = "".join(ch for ch in s if ch.isdigit() or ch == "-")
    return s

rx["DrugID_clean"] = rx["DrugID"].map(clean_id)

unique_ids = (
    pd.Series(sorted(rx["DrugID_clean"].dropna().unique()), name="DrugID_clean")
    .astype(str)
)

# Deterministic weighted assignment via hashing
probs = np.array(probs, dtype=float)
probs = probs / probs.sum()
edges = np.cumsum(probs)

seed = 42
h = pd.util.hash_pandas_object(unique_ids, index=False).astype("uint64")
x = ((h ^ np.uint64(seed)) % np.uint64(10**9)).astype("float64") / 1e9

idx = np.searchsorted(edges, x, side="right")
idx = np.clip(idx, 0, len(classes) - 1)

drug_class_map = pd.DataFrame({
    "DrugID_clean": unique_ids.values,
    "pharm_classes": np.array(classes, dtype=object)[idx]
})

# Export
drug_class_map_path = PROJECT_ROOT / "drug_class_map.csv"
drug_class_map.to_csv(drug_class_map_path, index=False)

print("Wrote:", drug_class_map_path)
print(drug_class_map.head())

Wrote: /Users/selenadavis/PythonProject/Notebooks/drug_class_map.csv
   DrugID_clean                                      pharm_classes
0  0001-0001-11  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...
1  0002-0002-22  Cytochrome P450 2D6 Inhibitors, Serotonin Reup...
2  0003-0003-33                Macrolide Antimicrobial, Macrolides
3  0004-0004-44  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...
4  0005-0005-55  Anti-epileptic Agent, Decreased Central Nervou...


In [7]:
# --- 1) Forecast demand & prescription volume by class per day (Power BI export) ---
import pandas as pd
import numpy as np

rx = pd.read_csv("prescriptions.csv", low_memory=False)
pred = pd.read_csv("predictions_daily_class.csv", low_memory=False)
mapping = pd.read_csv("bi_outputs/drug_class_map.csv", low_memory=False)

def clean_id(s):
    s = str(s)
    s = s.replace("\u200b", "").replace("\ufeff", "").strip()
    s = "".join(ch for ch in s if ch.isdigit() or ch == "-")
    return s

# Normalize mapping join key
# Support either DrugID_clean or DrugID in mapping
if "DrugID_clean" in mapping.columns:
    mapping["DrugID_clean"] = mapping["DrugID_clean"].map(clean_id)
elif "DrugID" in mapping.columns:
    mapping["DrugID_clean"] = mapping["DrugID"].map(clean_id)
else:
    raise ValueError(f"drug_class_map.csv must contain DrugID_clean or DrugID. Got: {list(mapping.columns)}")

if "pharm_classes" not in mapping.columns:
    raise ValueError(f"drug_class_map.csv must contain pharm_classes. Got: {list(mapping.columns)}")

# Clean rx keys + dates
if "DrugID" not in rx.columns:
    raise ValueError("prescriptions.csv must contain DrugID")

rx["DrugID_clean"] = rx["DrugID"].map(clean_id)

date_col = "FillDate" if "FillDate" in rx.columns else ("Date" if "Date" in rx.columns else None)
if date_col is None:
    raise ValueError("prescriptions.csv needs FillDate or Date")

rx["Date"] = pd.to_datetime(rx[date_col], errors="coerce").dt.date
rx = rx.dropna(subset=["Date"])  # remove bad date rows

# Merge class onto rx
rx_mapped = rx.merge(
    mapping[["DrugID_clean", "pharm_classes"]],
    on="DrugID_clean",
    how="left"
)
rx_mapped["pharm_classes"] = rx_mapped["pharm_classes"].fillna("Unmapped")

print("Unmapped rows:", int(rx_mapped["pharm_classes"].eq("Unmapped").sum()), "of", len(rx_mapped))

# Class-level actuals by day
# Count Rx (unique RxID if available, otherwise row count)
if "RxID" in rx_mapped.columns:
    rx_by_class_day = (
        rx_mapped.groupby(["Date", "pharm_classes"], as_index=False)
                 .agg(Actual_Rx=("RxID", "nunique"))
    )
else:
    rx_by_class_day = (
        rx_mapped.groupby(["Date", "pharm_classes"], as_index=False)
                 .size()
                 .rename(columns={"size": "Actual_Rx"})
    )

# Drop Unmapped
rx_by_class_day = rx_by_class_day[rx_by_class_day["pharm_classes"] != "Unmapped"].copy()

rx_by_class_day.to_csv("bi_outputs/prescription_volume_by_class_day.csv", index=False)
print("Wrote bi_outputs/prescription_volume_by_class_day.csv", rx_by_class_day.shape)

# Total actuals by day
if "RxID" in rx_mapped.columns:
    rx_by_day = (
        rx_mapped.groupby("Date", as_index=False)
                 .agg(TotalPrescriptions=("RxID", "nunique"))
    )
else:
    rx_by_day = (
        rx_mapped.groupby("Date", as_index=False)
                 .size()
                 .rename(columns={"size": "TotalPrescriptions"})
    )

rx_by_day.to_csv("bi_outputs/prescription_volume_by_day.csv", index=False)
print("Wrote bi_outputs/prescription_volume_by_day.csv", rx_by_day.shape)

# Forecast export (standard schema)
if "Date" not in pred.columns:
    raise ValueError(f"predictions_daily_class.csv must contain Date. Got: {list(pred.columns)}")

# Support common demand column names
demand_col = "Demand" if "Demand" in pred.columns else ("ForecastQty" if "ForecastQty" in pred.columns else None)
if demand_col is None:
    raise ValueError(f"predictions_daily_class.csv must contain Demand or ForecastQty. Got: {list(pred.columns)}")

if "DrugClass" not in pred.columns:
    raise ValueError(f"predictions_daily_class.csv must contain DrugClass. Got: {list(pred.columns)}")

pred_out = pred.copy()
pred_out["Date"] = pd.to_datetime(pred_out["Date"], errors="coerce").dt.date
pred_out = pred_out.dropna(subset=["Date"])
pred_out = pred_out.rename(columns={"DrugClass": "pharm_classes", demand_col: "ForecastQty"})
pred_out["pharm_classes"] = pred_out["pharm_classes"].astype(str).str.strip()

pred_out["ForecastQty"] = pd.to_numeric(pred_out["ForecastQty"], errors="coerce").fillna(0.0)
pred_out["ForecastQty"] = pred_out["ForecastQty"].clip(lower=0)

# Per-day scaling so forecast totals match actual totals
actual_totals = rx_by_day.rename(columns={"TotalPrescriptions": "ActualTotal"}).copy()

pred_totals = (
    pred_out.groupby("Date", as_index=False)
            .agg(PredTotal=("ForecastQty", "sum"))
)

scale = actual_totals.merge(pred_totals, on="Date", how="left")
scale["PredTotal"] = pd.to_numeric(scale["PredTotal"], errors="coerce").fillna(0.0)

# If PredTotal is 0, keep factor 1.0
scale["ScaleFactor"] = np.where(scale["PredTotal"] > 0, scale["ActualTotal"] / scale["PredTotal"], 1.0)

pred_scaled = pred_out.merge(scale[["Date", "ScaleFactor"]], on="Date", how="left")
pred_scaled["ScaleFactor"] = pred_scaled["ScaleFactor"].fillna(1.0)

pred_scaled["ForecastQty"] = (
    (pred_scaled["ForecastQty"] * pred_scaled["ScaleFactor"])
    .round()
    .clip(lower=0)
    .astype(int)
)

pred_out_final = pred_scaled[["Date", "pharm_classes", "ForecastQty"]].copy()

pred_out_final.to_csv("bi_outputs/forecast_demand_by_class_day.csv", index=False)
print("Wrote bi_outputs/forecast_demand_by_class_day.csv", pred_out_final.shape)
print(pred_out_final.head())

# Sanity prints to catch "all 9" style bugs
print("Actual_Rx describe:\n", rx_by_class_day["Actual_Rx"].describe())
print("ForecastQty describe:\n", pred_out_final["ForecastQty"].describe())
print("Unique dates (actual):", rx_by_class_day["Date"].nunique(), "Unique classes (actual):", rx_by_class_day["pharm_classes"].nunique())
print("Unique dates (forecast):", pred_out_final["Date"].nunique(), "Unique classes (forecast):", pred_out_final["pharm_classes"].nunique())

Unmapped rows: 0 of 120000
Wrote bi_outputs/prescription_volume_by_class_day.csv (10961, 3)
Wrote bi_outputs/prescription_volume_by_day.csv (731, 2)
Wrote bi_outputs/forecast_demand_by_class_day.csv (11680, 3)
         Date                                      pharm_classes  ForecastQty
0  2023-01-01  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...           28
1  2023-01-02  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...           27
2  2023-01-03  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...           14
3  2023-01-04  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...           20
4  2023-01-05  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...           16
Actual_Rx describe:
 count    10961.000000
mean        10.947906
std          4.826641
min          1.000000
25%          7.000000
50%         10.000000
75%         14.000000
max         36.000000
Name: Actual_Rx, dtype: float64
ForecastQty describe:
 count    11680.000000
mean        10.253168
std         13.515710
min 

In [8]:
# --- 2) Inventory + reference mapping (DrugName -> DrugClass) ---
if P_INV is None:
    raise FileNotFoundError("inventory.csv not found (Notebook1 output).")

inv = pd.read_csv(P_INV, low_memory=False)

# Standardize key columns
inv = rename_any(inv, ["OnHand","StockLevel","QtyOnHand","InventoryQty","Qty"], "OnHand")
inv = rename_any(inv, ["DrugName","drug_name","Medication","Drug","Name"], "DrugName")
inv = rename_any(inv, ["UnitCost","Cost","AvgCost"], "UnitCost")

for col, default in [("OnHand", 0.0), ("UnitCost", 0.0)]:
    if col not in inv.columns:
        inv[col] = default
    inv[col] = pd.to_numeric(inv[col], errors="coerce").fillna(default)

if "DrugName" not in inv.columns:
    inv["DrugName"] = "Unknown"

inv["DrugName_norm"] = inv["DrugName"].map(norm_name)

# Build a mapping table
if P_REF is not None:
    ref = pd.read_csv(P_REF, low_memory=False)
    ref = rename_any(ref, ["DrugName","drug_name","Name"], "DrugName")
    ref = rename_any(ref, ["DrugClass","Pharm_Classes","Class","pharm_classes"], "DrugClass")

    if "DrugName" not in ref.columns or "DrugClass" not in ref.columns:
        print("ref_drugs.csv found, but missing DrugName/DrugClass columns. Mapping disabled.")
        ref_map = pd.DataFrame({"DrugName_norm": [], "DrugClass": []})
    else:
        ref["DrugName_norm"] = ref["DrugName"].map(norm_name)
        ref_map = (ref[["DrugName_norm","DrugClass"]]
                   .dropna(subset=["DrugName_norm"])
                   .drop_duplicates(subset=["DrugName_norm"]))
else:
    ref_map = pd.DataFrame({"DrugName_norm": [], "DrugClass": []})

print("ref_map rows:", len(ref_map))

# Save reference map
ref_map.to_csv("ref_drugname_class_map.csv", index=False)
print("Wrote ref_drugname_class_map.csv", len(ref_map))

inv_mapped = inv.merge(ref_map, on="DrugName_norm", how="left").rename(columns={"DrugClass":"pharm_classes"})
inv_mapped["pharm_classes"] = inv_mapped["pharm_classes"].fillna("Unmapped")
inv_mapped.to_csv("inventory_mapped.csv", index=False)
print("Wrote inventory_mapped.csv", len(inv_mapped))

ref_map rows: 14312
Wrote ref_drugname_class_map.csv 14312
Wrote inventory_mapped.csv 220


In [9]:
# Scale forecasted with actual
forecast = pd.read_csv("bi_outputs/forecast_demand_by_class_day.csv")
actuals = pd.read_csv("bi_outputs/prescription_volume_by_class_day.csv")

# Ensure dates align
forecast["Date"] = pd.to_datetime(forecast["Date"]).dt.date
actuals["Date"] = pd.to_datetime(actuals["Date"]).dt.date

# Totals per day
daily_actual = (
    actuals.groupby("Date", as_index=False)
    .agg(ActualTotal=("Actual_Rx", "sum"))
)

daily_forecast = (
    forecast.groupby("Date", as_index=False)
    .agg(ForecastTotal=("ForecastQty", "sum"))
)

# Scale factor per day
scale = daily_actual.merge(daily_forecast, on="Date", how="left")
scale["ForecastTotal"] = pd.to_numeric(scale["ForecastTotal"], errors="coerce").fillna(0)

scale["ScaleFactor"] = np.where(
    scale["ForecastTotal"] > 0,
    scale["ActualTotal"] / scale["ForecastTotal"],
    1.0
)

# Apply scaling
forecast = forecast.merge(scale[["Date", "ScaleFactor"]], on="Date", how="left")
forecast["ForecastQty"] = (forecast["ForecastQty"] * forecast["ScaleFactor"]).round().clip(lower=0).astype(int)
forecast = forecast.drop(columns=["ScaleFactor"])

forecast.to_csv("bi_outputs/forecast_demand_by_class_day.csv", index=False)

print("Daily scaling applied.")
print("Actual total:", daily_actual["ActualTotal"].sum())
print("Forecast total after scaling:", forecast["ForecastQty"].sum())
print("ScaleFactor stats:")
print(scale["ScaleFactor"].describe())

Daily scaling applied.
Actual total: 120000
Forecast total after scaling: 119764
ScaleFactor stats:
count    731.000000
mean       1.000711
std        0.005338
min        0.983051
25%        1.000000
50%        1.000000
75%        1.005780
max        1.014184
Name: ScaleFactor, dtype: float64


In [10]:
# --- 3) Inventory turnover (class + overall) ---

# Inventory summarized by class
inv_m = inv.merge(ref_map, on="DrugName_norm", how="left")
inv_m["Pharm_Classes"] = inv_m["DrugClass"].fillna("Unmapped").astype(str).str.strip()

inv_class = (inv_m.groupby("Pharm_Classes", as_index=False)
             .agg(OnHand=("OnHand","sum"),
                  UnitCost=("UnitCost","mean")))
inv_class["InvValue"] = inv_class["OnHand"] * inv_class["UnitCost"]

# Rx annualized by class
rx_class = None

if P_RX is not None and Path(P_RX).exists():
    rx_df = pd.read_csv(P_RX, low_memory=False, dtype={"DrugID":"string"})
    rx_df = rename_any(rx_df, ["DrugName","drug_name","Medication","Drug","Name"], "DrugName")
    rx_cols_lower = {c.lower(): c for c in rx_df.columns}
    rx_date_col = rx_cols_lower.get("filldate") or rx_cols_lower.get("refilldate") or rx_cols_lower.get("date")

    if rx_date_col is not None:
        rx_df[rx_date_col] = pd.to_datetime(rx_df[rx_date_col], errors="coerce")

    if "DrugName" not in rx_df.columns:
        rx_df["DrugName"] = "Unknown"

    rx_df["DrugName_norm"] = rx_df["DrugName"].map(norm_name)
    rx_m = rx_df.merge(ref_map, on="DrugName_norm", how="left")
    rx_m["Pharm_Classes"] = rx_m["DrugClass"].fillna("Unmapped").astype(str).str.strip()

    if rx_date_col is not None:
        rx_m = rx_m.dropna(subset=[rx_date_col])

    if len(rx_m) and rx_date_col is not None:
        span_days = (rx_m[rx_date_col].max() - rx_m[rx_date_col].min()).days + 1
        annual_factor_rx = 365.0 / max(1, span_days)
        rx_class = (rx_m.groupby("Pharm_Classes", as_index=False)
                    .agg(RxCount=("DrugName_norm","size")))
        rx_class["AnnualizedRx"] = rx_class["RxCount"] * annual_factor_rx

if rx_class is None:
    rx_class = (demand_by_class[["Pharm_Classes","AnnualizedRx"]].copy()
                if demand_by_class is not None else pd.DataFrame({"Pharm_Classes": [], "AnnualizedRx": []}))

turn = inv_class.merge(rx_class[["Pharm_Classes","AnnualizedRx"]], on="Pharm_Classes", how="outer")

for col in ["OnHand","UnitCost","InvValue","AnnualizedRx"]:
    if col not in turn.columns:
        turn[col] = 0
    turn[col] = pd.to_numeric(turn[col], errors="coerce").fillna(0)

turn["Turnover"] = np.where(turn["OnHand"] > 0, turn["AnnualizedRx"] / turn["OnHand"], np.nan)
turn["Turnover_Value"] = np.where(turn["InvValue"] > 0,
                                  (turn["AnnualizedRx"] * turn["UnitCost"]) / turn["InvValue"],
                                  np.nan)

overall_onhand = float(turn["OnHand"].sum())
overall_annual = float(turn["AnnualizedRx"].sum())
overall_turnover = (overall_annual / overall_onhand) if overall_onhand > 0 else np.nan

overall = pd.DataFrame([{
    "Pharm_Classes": "__OVERALL__",
    "OnHand": overall_onhand,
    "UnitCost": np.nan,
    "InvValue": float(turn["InvValue"].sum()),
    "AnnualizedRx": overall_annual,
    "Turnover": overall_turnover,
    "Turnover_Value": np.nan
}])

turn = pd.concat([overall, turn], ignore_index=True)
turn.to_csv(OUT_DIR / "inventory_turnover.csv", index=False)

print("Wrote:", OUT_DIR / "inventory_turnover.csv")

# Copy staffing exports into BI folder
if P_STAFF_HOURLY is not None:
    pd.read_csv(P_STAFF_HOURLY, low_memory=False).to_csv(OUT_DIR / "projected_staffing_by_hour.csv", index=False)

if P_STAFF_COVERAGE is not None:
    pd.read_csv(P_STAFF_COVERAGE, low_memory=False).to_csv(OUT_DIR / f"{Path(P_STAFF_COVERAGE).name}", index=False)

Wrote: bi_outputs/inventory_turnover.csv


In [11]:
# --- 4) Rx Revenue by Insurance Type ---

OUT_DIR = Path("bi_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load sales first
sales_path = Path("sales.csv")
rx_path = Path("prescriptions.csv")

if sales_path.exists():
    sales = pd.read_csv(sales_path, low_memory=False)

    # Expected: Date, PaymentType, Revenue, RxID
    if "Date" not in sales.columns:
        raise ValueError(f"sales.csv missing 'Date'. Columns: {list(sales.columns)}")
    if "PaymentType" not in sales.columns:
        raise ValueError(f"sales.csv missing 'PaymentType'. Columns: {list(sales.columns)}")
    if "Revenue" not in sales.columns:
        raise ValueError(f"sales.csv missing 'Revenue'. Columns: {list(sales.columns)}")

    sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce").dt.date
    sales = sales.dropna(subset=["Date"])

    sales["InsuranceType"] = sales["PaymentType"].astype(str).str.strip().replace({"": "Unknown"}).fillna("Unknown")
    sales["Revenue"] = pd.to_numeric(sales["Revenue"], errors="coerce").fillna(0.0)

    # RxCount: count unique RxID if present, else count rows
    if "RxID" in sales.columns:
        grouped = (
            sales.groupby(["Date", "InsuranceType"], as_index=False)
                 .agg(
                     Revenue=("Revenue", "sum"),
                     RxCount=("RxID", "nunique")
                 )
        )
    else:
        grouped = (
            sales.groupby(["Date", "InsuranceType"], as_index=False)
                 .agg(
                     Revenue=("Revenue", "sum"),
                     RxCount=("Revenue", "size")
                 )
        )

else:
    # --- Fallback: derive revenue by insurance from prescriptions.csv ---
    if not rx_path.exists():
        raise FileNotFoundError("Need sales.csv or prescriptions.csv in this folder.")

    rx = pd.read_csv(rx_path, low_memory=False)

    date_col = "FillDate" if "FillDate" in rx.columns else ("Date" if "Date" in rx.columns else None)
    if date_col is None:
        raise ValueError(f"prescriptions.csv missing FillDate/Date. Columns: {list(rx.columns)}")

    if "InsuranceType" not in rx.columns:
        raise ValueError("prescriptions.csv missing InsuranceType (needed for payer mix).")

    cost_col = "Cost" if "Cost" in rx.columns else None
    if cost_col is None:
        raise ValueError("prescriptions.csv missing Cost (needed for revenue proxy).")

    rx["Date"] = pd.to_datetime(rx[date_col], errors="coerce").dt.date
    rx = rx.dropna(subset=["Date"])

    rx["InsuranceType"] = rx["InsuranceType"].astype(str).str.strip().replace({"": "Unknown"}).fillna("Unknown")
    rx["Revenue"] = pd.to_numeric(rx[cost_col], errors="coerce").fillna(0.0)

    if "RxID" in rx.columns:
        grouped = (
            rx.groupby(["Date", "InsuranceType"], as_index=False)
              .agg(
                  Revenue=("Revenue", "sum"),
                  RxCount=("RxID", "nunique")
              )
        )
    else:
        grouped = (
            rx.groupby(["Date", "InsuranceType"], as_index=False)
              .agg(
                  Revenue=("Revenue", "sum"),
                  RxCount=("Revenue", "size")
              )
        )

# Final cleanup
grouped["Revenue"] = grouped["Revenue"].round(2)
grouped["RxCount"] = grouped["RxCount"].astype(int)

out_path = OUT_DIR / "revenue_rx_by_insurance_day.csv"
grouped.to_csv(out_path, index=False)

print("Wrote:", out_path, grouped.shape)
print(grouped.head(10))
print("Revenue describe:\n", grouped["Revenue"].describe())
print("RxCount describe:\n", grouped["RxCount"].describe())
print("Unique dates:", grouped["Date"].nunique(), "Unique insurance types:", grouped["InsuranceType"].nunique())

Wrote: bi_outputs/revenue_rx_by_insurance_day.csv (2924, 4)
         Date InsuranceType  Revenue  RxCount
0  2023-01-01          Cash  1735.75       13
1  2023-01-01    Commercial  6396.06       50
2  2023-01-01      Medicaid  6204.93       52
3  2023-01-01      Medicare  5821.46       55
4  2023-01-02          Cash  1599.57       14
5  2023-01-02    Commercial  5562.19       48
6  2023-01-02      Medicaid  4237.27       37
7  2023-01-02      Medicare  4724.00       39
8  2023-01-03          Cash  2555.49       16
9  2023-01-03    Commercial  5993.63       50
Revenue describe:
 count     2924.000000
mean      5179.483478
std       2211.425269
min        412.850000
25%       3357.685000
50%       5687.830000
75%       6817.960000
max      11006.560000
Name: Revenue, dtype: float64
RxCount describe:
 count    2924.000000
mean       41.039672
std        17.219294
min         5.000000
25%        26.000000
50%        45.000000
75%        54.000000
max        80.000000
Name: RxCount, dtype: 

In [12]:
# --- 5) Revenue by insurance type ---
if P_RX is None:
    raise FileNotFoundError("prescriptions.csv not found (Notebook1 output).")

rx = pd.read_csv(P_RX, low_memory=False)

rx_cols_lower = {c.lower(): c for c in rx.columns}
date_col = rx_cols_lower.get("filldate") or rx_cols_lower.get("refilldate") or rx_cols_lower.get("date")
if date_col is None:
    raise ValueError("prescriptions.csv needs FillDate or RefillDate (or Date)")

rx[date_col] = pd.to_datetime(rx[date_col], errors="coerce")

if "Cost" not in rx.columns:
    raise ValueError("prescriptions.csv needs Cost for revenue")

rx["Revenue"] = pd.to_numeric(rx["Cost"], errors="coerce").fillna(0)

if "InsuranceType" not in rx.columns:
    if P_PATIENTS is None:
        raise FileNotFoundError("patients.csv not found, but prescriptions.csv has no InsuranceType.")
    patients = pd.read_csv(P_PATIENTS, low_memory=False)
    if "PatientID" not in rx.columns or "PatientID" not in patients.columns or "InsuranceType" not in patients.columns:
        raise ValueError("Need InsuranceType in prescriptions.csv or (patients.csv with PatientID + InsuranceType).")
    rx = rx.merge(patients[["PatientID","InsuranceType"]], on="PatientID", how="left")

rx["InsuranceType"] = rx["InsuranceType"].fillna("Unknown").astype(str).str.strip()

base = rx.dropna(subset=[date_col]).copy()
base["Date"] = base[date_col].dt.date.astype(str)

revenue_by_ins = (
    base.groupby(["Date","InsuranceType"], as_index=False)
        .agg(
            Revenue=("Revenue","sum"),
            RxCount=("Revenue","size"),
        )
)

revenue_by_ins.to_csv(OUT_DIR / "revenue_by_insurance_type.csv", index=False)
print("Wrote:", OUT_DIR / "revenue_by_insurance_type.csv", "rows:", len(revenue_by_ins))

Wrote: bi_outputs/revenue_by_insurance_type.csv rows: 2924


In [13]:
# --- 6) date_table.csv for Power BI ---
import pandas as pd

# Explicitly define the files that contain valid dates
files_with_dates = [
    "bi_outputs/prescription_volume_by_day.csv",
    "bi_outputs/forecast_demand_by_class_day.csv"
]

min_dt, max_dt = None, None

for file in files_with_dates:
    df = pd.read_csv(file)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    lo, hi = df["Date"].min(), df["Date"].max()

    min_dt = lo if (min_dt is None or lo < min_dt) else min_dt
    max_dt = hi if (max_dt is None or hi > max_dt) else max_dt

# Safety fallback
if min_dt is None or max_dt is None:
    min_dt = pd.Timestamp("2023-01-01")
    max_dt = pd.Timestamp.today().normalize()

dates = pd.date_range(min_dt.normalize(), max_dt.normalize(), freq="D")

date_table = pd.DataFrame({"Date": dates})
date_table["Year"] = date_table["Date"].dt.year
date_table["Month"] = date_table["Date"].dt.month
date_table["MonthName"] = date_table["Date"].dt.strftime("%b")
date_table["Quarter"] = date_table["Date"].dt.quarter
date_table["Week"] = date_table["Date"].dt.isocalendar().week.astype(int)
date_table["Day"] = date_table["Date"].dt.day
date_table["DayName"] = date_table["Date"].dt.strftime("%a")
date_table["IsWeekend"] = date_table["Date"].dt.weekday >= 5

date_table.to_csv("bi_outputs/date_table.csv", index=False)

print("Wrote bi_outputs/date_table.csv range:",
      min_dt.date(), "to", max_dt.date())

Wrote bi_outputs/date_table.csv range: 2023-01-01 to 2024-12-31


In [14]:
# --- 7) KPI summary ---
OUT_KPI = OUT_DIR / "kpi_summary.csv"

def load_out_csv(filename):
    p = OUT_DIR / filename
    return pd.read_csv(p) if p.exists() else None

def pick_numeric_col(df, preferred_names):
    if df is None:
        return None
    cols = {c.lower(): c for c in df.columns}
    for name in preferred_names:
        if name.lower() in cols:
            return cols[name.lower()]
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return num_cols[0] if num_cols else None

def safe_sum(df, col):
    if df is None or col is None or col not in df.columns:
        return None
    return float(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

def get_overall_turnover(turnover_df):
    if turnover_df is None or len(turnover_df) == 0:
        return None
    if "Pharm_Classes" in turnover_df.columns and "Turnover" in turnover_df.columns:
        mask = turnover_df["Pharm_Classes"].astype(str).str.strip().eq("__OVERALL__")
        if mask.any():
            val = pd.to_numeric(turnover_df.loc[mask, "Turnover"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            return float(val.iloc[0]) if len(val) else None
    if "AnnualizedRx" in turnover_df.columns and "OnHand" in turnover_df.columns:
        annual = pd.to_numeric(turnover_df["AnnualizedRx"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).sum()
        onhand = pd.to_numeric(turnover_df["OnHand"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).sum()
        return float(annual / onhand) if onhand > 0 else None
    return None

def fmt(val, kind="number"):
    if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
        return None
    if kind == "int":
        return int(round(val))
    if kind == "currency":
        return round(val, 2)
    return round(val, 4)

# Load pipeline outputs
forecast_df = load_out_csv("forecast_demand_by_class_day.csv")
staff_df = load_out_csv("projected_staffing_by_hour.csv")
turnover_df = load_out_csv("inventory_turnover.csv")
revenue_df = load_out_csv("revenue_by_insurance_type.csv")

# Total forecast demand
demand_col = pick_numeric_col(forecast_df, ["ForecastQty", "Demand", "Quantity", "Qty"])
total_forecast = safe_sum(forecast_df, demand_col)

# Staffing
total_staff = None
staff_col = None

if staff_df is not None and len(staff_df) > 0:
    # Clean any #ERROR-like junk by coercing to numeric
    for c in ["Required", "Scheduled", "RequiredStaff", "ScheduledStaff", "HoursWorked", "Staffing Delta", "StaffingDelta"]:
        if c in staff_df.columns:
            staff_df[c] = pd.to_numeric(staff_df[c], errors="coerce")

    # Prefer Scheduled if available (represents planned staffing)
    if "Scheduled" in staff_df.columns:
        staff_col = "Scheduled"
        total_staff = float(pd.to_numeric(staff_df["Scheduled"], errors="coerce").fillna(0).sum())
    elif "ScheduledStaff" in staff_df.columns:
        staff_col = "ScheduledStaff"
        total_staff = float(pd.to_numeric(staff_df["ScheduledStaff"], errors="coerce").fillna(0).sum())
    else:
        # Fallbacks
        staff_col = pick_numeric_col(staff_df, ["RequiredStaff", "HoursWorked", "Required", "Scheduled"])
        total_staff = safe_sum(staff_df, staff_col)

# Turnover
avg_turnover = get_overall_turnover(turnover_df)

# Revenue
rev_col = pick_numeric_col(revenue_df, ["Revenue", "TotalRevenue", "Sales", "Amount"])
total_revenue = safe_sum(revenue_df, rev_col)

rows = [
    {
        "metric": "Total Forecast Demand",
        "value": fmt(total_forecast, "int") if total_forecast is not None else "Needs input",
        "unit": "units",
        "source_file": "forecast_demand_by_class_day.csv",
        "notes": f"Summed '{demand_col}'" if demand_col else "Could not detect demand column.",
    },
    {
        "metric": "Total Projected Staffing",
        "value": fmt(total_staff, "int") if total_staff is not None else "Needs input",
        "unit": "staff-units",
        "source_file": "projected_staffing_by_hour.csv" if staff_df is not None else "",
        "notes": f"Summed '{staff_col}'" if staff_col else "Staffing export not found in bi_outputs.",
    },
    {
        "metric": "Average Inventory Turnover",
        "value": fmt(avg_turnover, "number") if avg_turnover is not None else "Needs input",
        "unit": "turns",
        "source_file": "inventory_turnover.csv",
        "notes": "Uses '__OVERALL__' if present; else sum(AnnualizedRx)/sum(OnHand).",
    },
    {
        "metric": "Total Revenue",
        "value": fmt(total_revenue, "currency") if total_revenue is not None else "Needs input",
        "unit": "USD",
        "source_file": "revenue_by_insurance_type.csv",
        "notes": f"Summed '{rev_col}'" if total_revenue is not None else "revenue_by_insurance_type.csv not found in bi_outputs.",
    },
]

kpi_df = pd.DataFrame(rows)
kpi_df.to_csv(OUT_KPI, index=False)

print("Wrote:", OUT_KPI)
print(kpi_df)

Wrote: bi_outputs/kpi_summary.csv
                       metric         value         unit  \
0       Total Forecast Demand  1.197640e+05        units   
1    Total Projected Staffing  2.661300e+04  staff-units   
2  Average Inventory Turnover  2.178000e-01        turns   
3               Total Revenue  1.514481e+07          USD   

                        source_file  \
0  forecast_demand_by_class_day.csv   
1    projected_staffing_by_hour.csv   
2            inventory_turnover.csv   
3     revenue_by_insurance_type.csv   

                                               notes  
0                               Summed 'ForecastQty'  
1                            Summed 'ScheduledStaff'  
2  Uses '__OVERALL__' if present; else sum(Annual...  
3                                   Summed 'Revenue'  
